In [44]:
import pandas as pd
import networkx as nx
import numpy as np
import seaborn

In [31]:
def under_model(G,reps):
    old_clustering=nx.clustering(G).values()
    degree=G.degree().values()

    clust_mat=np.zeros((reps,G.number_of_nodes()))
    for i in xrange(reps):
        C=nx.configuration_model(degree)
        C=nx.Graph(C) #remove parallel edges
        C.remove_edges_from(C.selfloop_edges()) #remove self-loops
        clustering=nx.clustering(C)
        clust_mat[i]=clustering.values()
    means=np.mean(clust_mat, axis=0) #mean over iterations for each node
    per25=np.percentile(clust_mat, 25, axis=0)
    per75=np.percentile(clust_mat, 75, axis=0) 
    return [old_clustering-means, old_clustering-per25, old_clustering-per75]

In [32]:
def under_weighted_model(G,reps):
    lmbda=1
    old_clustering=nx.clustering(G).values()
    degree=G.degree().values()

    clust_mat=np.zeros((reps,G.number_of_nodes()))
    for i in xrange(reps):
        C=nx.configuration_model(degree)
        C=nx.Graph(C) #remove parallel edges
        C.remove_edges_from(C.selfloop_edges()) #remove self-loops
        for e in C.edges():
            C.edge[e[0]][e[1]]['weight']=np.random.poisson()+1 #start with 1
        clustering=nx.clustering(C, weight='weight')
        clust_mat[i]=clustering.values()
    means=np.mean(clust_mat, axis=0) #mean over iterations for each node
    per25=np.percentile(clust_mat, 25, axis=0)
    per75=np.percentile(clust_mat, 75, axis=0) 
    return [old_clustering-means, old_clustering-per25, old_clustering-per75]

In [33]:
def configModelGraphs(G,reps):
    degree=G.degree().values()
    graphs=[]
    for i in xrange(reps):
        C=nx.configuration_model(degree)
        C=nx.Graph(C) #remove parallel edges
        C.remove_edges_from(C.selfloop_edges()) #remove self-loops
        graphs.append(C)
    return graphs

In [34]:
def weightConfigModelGraphs(G,reps):
    degree=G.degree().values()
    graphs=[]
    for i in xrange(reps):
        C=nx.configuration_model(degree)
        C=nx.Graph(C) #remove parallel edges
        C.remove_edges_from(C.selfloop_edges()) #remove self-loops
        for e in C.edges():
            C.edge[e[0]][e[1]]['weight']=np.random.poisson()+1 #start with 1
        graphs.append(C)
    return graphs

In [38]:
posts=pd.read_csv('/Users/Ish/Desktop/data_export_posts_dib.csv',sep=',', index_col=0)

In [40]:
comments=pd.read_csv('/Users/Ish/Desktop/data_export_comments_dib.csv',sep=',', index_col=0)

In [48]:
comments.parent_id[:5]

1     t3_1x812a
2    t1_cf8pqsr
3     t3_1x8ivn
4    t1_cf98aja
5    t1_cf8tfiu
Name: parent_id, dtype: object

In [55]:
comments['parent_post']=comments.parent_id.apply(lambda x: x[3:])

In [56]:
comments['parent_post_type']=comments.parent_id.apply(lambda x: x[:2])

In [59]:
comments.head()

,id,ups,downs,score,author,edited,gilded,link_id,parent_id,subreddit,created_utc,created_time,parent_post,parent_post_type
1,cf91bfl,-1,0,-1,a_until_z,f,0,t3_1x812a,t3_1x812a,dataisbeautiful,1391737883,2014-02-07 01:51:23,1x812a,t3
2,cf94xpo,1,0,1,kaiden333,f,0,t3_1x6o6k,t1_cf8pqsr,dataisbeautiful,1391745946,2014-02-07 04:05:46,cf8pqsr,t1
3,cf97irz,22,0,22,Lyeranth,f,0,t3_1x8ivn,t3_1x8ivn,dataisbeautiful,1391752843,2014-02-07 06:00:43,1x8ivn,t3
4,cf98das,7,0,7,[deleted],f,0,t3_1x8ivn,t1_cf98aja,dataisbeautiful,1391755816,2014-02-07 06:50:16,cf98aja,t1
5,cf9bp15,1,0,1,[deleted],f,0,t3_1x6o6k,t1_cf8tfiu,dataisbeautiful,1391774646,2014-02-07 12:04:06,cf8tfiu,t1


In [97]:
comments['parent']=comments.apply(lambda x: x.author if x.parent_post_type=='t1' else 0, axis=1)

In [98]:
comments.parent.head()

1            0
2    kaiden333
3            0
4    [deleted]
5    [deleted]
Name: parent, dtype: object

In [99]:
origPosts=comments[comments.parent_post_type=='t3'].parent_post.unique()

In [108]:
origPosts[:10]

array(['1x812a', '1x8ivn', '1x9xeb', '1xaehi', '1xczc4', '1xfiry',
       '1xfr9p', '1xj3c5', '1xin4w', '1xixun'], dtype=object)

In [104]:
len(posts[posts.id.isin(origPosts)].author)

5784

In [106]:
len(comments[comments.parent==0])

35455

# t3 is a post, t1 is a comment


In [110]:
len(comments[comments.parent_post_type=='t1'])

87185

In [111]:
len(comments[comments.parent_post_type=='t3'])

35455

In [112]:
len(comments)

122640

In [113]:
Out[110]+Out[111]

122640

In [123]:
len(origPosts)

5828

In [125]:
len(posts[posts.id.isin(origPosts)])

5784

In [126]:
35455-5784

29671